In [1]:
import os
import random

In [3]:
# Define the folder containing the audio files
audio_folder = r"AudioFiles/085"

# Get a list of all WAVE files in the folder
all_songs = [f for f in os.listdir(audio_folder) if f.endswith(".wav")]

In [6]:
random.shuffle(all_songs)

num_pairs = min(10, len(all_songs) // 3)

test_triplets = [
    tuple(all_songs[i:i+3])
    for i in range(0, num_pairs * 3, 3)
]

In [7]:
# Background image path
background_image = "background.png"

In [8]:
SUPABASE_URL = "https://zpvhnhnmjitkskhrhalp.supabase.co"
SUPABASE_ANON_KEY = "sb_publishable_jrzABfRbbZN4BNrUmABKag_b3LlgY18"

In [9]:
html_content = f"""<!DOCTYPE html>
<html>
<head>
    <meta charset="UTF-8">
    <title>Music Similarity Test</title>

    <style>
        body {{
            font-family: Helvetica, Arial, sans-serif;
            background: url('{background_image}') no-repeat center center fixed;
            background-size: cover;
            color: white;
            max-width: 900px;
            margin: 0 auto;
            padding: 20px;
        }}

        input[type="text"] {{
            padding: 6px;
            width: 220px;
            margin-top: 4px;
            margin-bottom: 16px;
        }}

        audio {{
            display: block;
            margin-bottom: 12px;
        }}

        .test-block {{
            margin-bottom: 30px;
            padding: 16px;
            background: rgba(0, 0, 0, 0.35);
            border-radius: 10px;
        }}

        .submit-row {{
            margin-top: 24px;
        }}
    </style>

    <script src="https://cdn.jsdelivr.net/npm/@supabase/supabase-js@2"></script>

    <script>
        const SUPABASE_URL = "{SUPABASE_URL}";
        const SUPABASE_ANON_KEY = "{SUPABASE_ANON_KEY}";
        const supabaseClient = supabase.createClient(SUPABASE_URL, SUPABASE_ANON_KEY);

        async function saveResults(event) {{
            event.preventDefault();

            const consentGiven = document.getElementById("consent").checked;
            if (!consentGiven) {{
                alert("You must agree to participate before submitting.");
                return;
            }}

            const participantCode = document.getElementById("participant_code")?.value.trim() || null;

            let results = [];
            for (let i = 0; i < {num_pairs}; i++) {{
                const selected = document.querySelector('input[name="triplet' + i + '"]:checked');
                const choice = selected ? selected.value : "No selection";

                const reference = document.getElementById("reference_" + i).dataset.song;
                const songA = document.getElementById("songA_" + i).dataset.song;
                const songB = document.getElementById("songB_" + i).dataset.song;

                results.push({{
                    triplet: i + 1,
                    reference_song: reference,
                    song_a: songA,
                    song_b: songB,
                    chosen_song: choice
                }});
            }}

            const submitBtn = document.getElementById("submitBtn");
            submitBtn.disabled = true;
            submitBtn.value = "Submitting...";

            const {{ error }} = await supabaseClient
                .from("listening_results")
                .insert([{{
                    participant_code: participantCode,
                    consent: true,
                    test_version: "v1",
                    responses: results
                }}]);

            submitBtn.disabled = false;
            submitBtn.value = "Save Results";

            if (error) {{
                console.error("Supabase insert error:", error);
                alert("Submission failed. Check console.");
                return;
            }}

            alert("Thank you. Your responses were saved.");
            document.querySelector("form").reset();
        }}
    </script>
</head>
<body>
    <h1>Music Similarity Test</h1>

    <p>
        Participant code:<br>
        <input type="text" id="participant_code" placeholder="e.g. P001">
    </p>

    <form onsubmit="saveResults(event);">
"""

In [10]:
for i, (reference_song, song_a, song_b) in enumerate(test_triplets):
    html_content += f"""
        <div class="test-block">
            <h3>Test {i + 1}</h3>

            <p>Reference Song:</p>
            <audio controls id="reference_{i}" data-song="{reference_song}">
                <source src="AudioFiles/085/{reference_song}" type="audio/wav">
            </audio>

            <p>Song A:</p>
            <audio controls id="songA_{i}" data-song="{song_a}">
                <source src="AudioFiles/085/{song_a}" type="audio/wav">
            </audio>

            <p>Song B:</p>
            <audio controls id="songB_{i}" data-song="{song_b}">
                <source src="AudioFiles/085/{song_b}" type="audio/wav">
            </audio>

            <p>Which song is more similar to the reference song?</p>

            <input type="radio" name="triplet{i}" value="A"> Song A<br>
            <input type="radio" name="triplet{i}" value="B"> Song B<br>
            <input type="radio" name="triplet{i}" value="None"> Neither, both are equally different<br>
        </div>
    """

In [11]:
html_content += """
        <p>
            <input type="checkbox" id="consent" required>
            I agree to participate in this study and allow my responses to be used for research purposes.
        </p>

        <div class="submit-row">
            <input type="submit" id="submitBtn" value="Save Results">
        </div>
    </form>

    <p style="font-size: 12px; margin-top: 20px;">
        This experiment is based on research from
        <a href="https://arxiv.org/abs/1612.01840" target="_blank" style="color: white;">
            arXiv:1612.01840
        </a>.
    </p>

    <p style="font-size: 12px;">
        The metadata is released under the Creative Commons Attribution 4.0 International License (CC BY 4.0).
        We do not hold the copyright on the audio and distribute it under the license chosen by the artist.
        The dataset is meant for research purposes.
    </p>
</body>
</html>
"""

In [12]:
with open("index.html", "w", encoding="utf-8") as f:
    f.write(html_content)

print("index.html generated successfully.")

index.html generated successfully.
